<a href="https://colab.research.google.com/github/santoprestandrea/bottle-anomaly-detection-project/blob/main/notebook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import torch
print(torch.cuda.is_available())
print(torch.__version__)
print(torch.cuda.get_device_name(0))

False
2.11.0+cpu


AssertionError: Torch not compiled with CUDA enabled

In [ ]:
import os
import glob
import random
import numpy as np
import cv2
import torch
import torch.nn as nn
import torchvision.transforms as T
from torch.utils.data import Dataset, DataLoader
from torchvision.models import resnet18, ResNet18_Weights
from sklearn.neighbors import NearestNeighbors
from sklearn.decomposition import PCA
from sklearn.metrics import roc_auc_score, roc_curve
from scipy.ndimage import gaussian_filter
import matplotlib.pyplot as plt
from tqdm import tqdm # barra di progresso -> se devo aspettare 30s, invece di aspettare nel vuoto vedo una barra che avanza

torch.manual_seed(42) # queste 3 righe servono per rendere i risultati riproducibili, in modo che ogni volta che eseguo il codice ottengo gli stessi risultati
np.random.seed(42)
random.seed(42)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {DEVICE}")

# Section 1 — The Problem: Industrial Anomaly Detection

## Problem Statement

In manufacturing, **quality control** is the process of identifying defective products before they reach customers. Traditionally this task is performed by human inspectors, an approach that is costly, slow, and prone to inconsistency — particularly for high-throughput production lines.

This project addresses the automation of quality control through Computer Vision. The goal is to detect **visual anomalies** (scratches, fractures, contamination, deformations) in product images, using a pipeline that requires no manual inspection.

---

## The Core Challenge: One-Class Learning

Standard supervised classification requires labeled examples of *both* classes — in this case, both normal and defective products. In a real industrial setting, however, defective items are rare by design: a well-functioning production line produces very few of them. As a result, collecting a balanced labeled dataset of defects is impractical.

This project instead adopts a **one-class learning** strategy:

> Learn a representation of what a *normal* product looks like. At test time, flag any image that deviates significantly from that representation as anomalous.

No defective images are used during training. The model is exposed only to normal samples, and anomaly detection emerges from measuring deviation from the learned normality.

---

## Approach

The pipeline implements a simplified version of **PatchCore** (Roth et al., 2022), structured in four stages:

1. **Feature Extraction** — A pre-trained ResNet18 backbone is used to extract spatial feature maps from each training image. Rather than using a global descriptor, features are extracted at the patch level, preserving spatial localization.
2. **Memory Bank Construction** — All patch-level feature vectors extracted from normal training images are stored in a memory bank, forming a complete reference of known normal local appearances.
3. **Anomaly Scoring** — For each test image, the distance of each patch descriptor to its nearest neighbors in the memory bank is computed. A high distance indicates that the local appearance is unusual with respect to the normal distribution.
4. **Post-processing** — Raw anomaly maps are smoothed, upsampled to the original image resolution, and thresholded to produce pixel-level defect segmentation masks.
---

## Dataset: MVTec Anomaly Detection

The **MVTec AD dataset** (Bergmann et al., 2019) is the standard benchmark for industrial anomaly detection research. It contains 15 categories of objects and textures, each with:

- A **training set** composed exclusively of defect-free images.
- A **test set** containing both normal images and images with multiple defect types.
- **Pixel-level ground truth masks** for all defective test images.
This project uses the **bottle** category:

| Split | Content | Count |
|-------|---------|-------|
| Train | Normal images only | ~209 |
| Test — normal | Defect-free bottles | ~20 |
| Test — defective | `broken_large`, `broken_small`, `contamination` | ~63 |

Images have an original resolution of 900×900 pixels and are resized to 224×224 for processing.


In [ ]:
# ── Mount Google Drive ────────────────────────────────────────────────────────
# In Google Colab, files saved locally disappear when the session ends.
# To use a dataset uploaded to Google Drive we "mount" it:
# this makes it accessible as if it were a local folder on the machine.
from google.colab import drive
drive.mount("/content/drive")  # /content/drive will be the access point to Drive


DATASET_ROOT = "/content/drive/MyDrive/bottle"

# ── Configuration ─────────────────────────────────────────────────────────────
# All images will be resized to 224x224 pixels.
# This is the standard input size for ResNet18 (the network we will use).
IMAGE_SIZE = 224

# 'assert' is a sanity check: if the condition is False, it raises an error with the given message.
# os.path.isdir() checks whether the path exists and is a directory.
# This lets us catch a wrong dataset path immediately.
assert os.path.isdir(DATASET_ROOT), f"Dataset not found at: {DATASET_ROOT}"
print(f"Dataset found: {DATASET_ROOT}")

# os.listdir() returns a list with the names of files and folders inside the given path
print("Contents:", os.listdir(DATASET_ROOT))

In [ ]:
# ── Build paths to the dataset sub-folders ────────────────────────────────────
# os.path.join() concatenates path segments correctly for the operating system
# (uses / on Linux/Mac, \ on Windows automatically)
train_dir   = os.path.join(DATASET_ROOT, "train", "good")   # normal training images
test_dir    = os.path.join(DATASET_ROOT, "test")             # root folder of the test set

# ── Count images per category ─────────────────────────────────────────────────
# List comprehension: [expression for item in list if condition]
# A compact way to build a filtered list from another list.
# Here: take sub-folder names from the test directory, excluding "good" (the normal class)
defect_types = sorted([d for d in os.listdir(test_dir) if d != "good"])
# sorted() orders the list alphabetically

# glob.glob() finds all files matching a pattern; "*.png" = every .png file
# len() counts how many elements are in the resulting list
n_train        = len(glob.glob(os.path.join(train_dir, "*.png")))
n_test_normal  = len(glob.glob(os.path.join(test_dir, "good", "*.png")))

# Count defective images by summing across every defect type
# This is a "generator expression": like a list comprehension but more memory-efficient
n_test_defect  = sum(
    len(glob.glob(os.path.join(test_dir, d, "*.png"))) for d in defect_types
)

print("── Dataset Statistics ─────────────────────")
print(f"  Train (normal only) : {n_train} images")
print(f"  Test  — normal      : {n_test_normal} images")
print(f"  Test  — defective   : {n_test_defect} images")
print(f"  Defect types        : {defect_types}")
print("───────────────────────────────────────────")

In [ ]:
# ── Image reading helper ──────────────────────────────────────────────────────
def read_rgb(path):
    # cv2.imread() reads an image from disk and returns it as a NumPy array
    # PROBLEM: OpenCV uses BGR (Blue-Green-Red) channel order instead of RGB
    # cv2.cvtColor() converts the image from BGR to RGB,
    # which is what matplotlib expects for correct colour display
    return cv2.cvtColor(cv2.imread(path), cv2.COLOR_BGR2RGB)

# ── Collect one sample image per category ─────────────────────────────────────
# Take the first normal image from the training set (index [0] after sorting)
samples = [sorted(glob.glob(os.path.join(train_dir, "*.png")))[0]]
titles  = ["NORMAL (train)"]

# Add one example for each defect type
for d in defect_types:
    p = sorted(glob.glob(os.path.join(test_dir, d, "*.png")))[0]
    samples.append(p)        # .append() adds one element to the end of the list
    titles.append(f"DEFECT: {d}")

# ── Build the multi-panel figure ──────────────────────────────────────────────
# plt.subplots(rows, cols) creates a grid of panels (axes)
# figsize=(width, height) sets the figure size in inches
fig, axes = plt.subplots(1, len(samples), figsize=(4 * len(samples), 4))

# zip() pairs elements from multiple iterables one-by-one:
# e.g. zip([a,b],[1,2]) -> [(a,1),(b,2)]
for ax, path, title in zip(axes, samples, titles):
    ax.imshow(read_rgb(path))   # display the image in the current panel
    ax.set_title(title, fontsize=10)
    ax.axis("off")              # hide axes and tick numbers around the image

# Overall title for the whole figure
plt.suptitle("MVTec AD — Bottle Category", fontsize=13, fontweight="bold")
plt.tight_layout()  # automatically optimise spacing between panels
plt.show()          # render and display the figure

# Section 2 — Preprocessing

## Overview

Raw images cannot be fed directly into a neural network. The network has strict requirements
on input format, spatial dimensions, and value range. The preprocessing pipeline transforms
each image through a fixed sequence of operations that resolve three fundamental issues:

- **Dimension mismatch** — dataset images are 900×900 pixels; ResNet18 expects 224×224.
- **Format mismatch** — raw NumPy arrays must be converted to PyTorch tensors with channels-first layout.
- **Scale mismatch** — pixel values in [0, 255] must be normalized to match the statistical distribution seen during ResNet18's training on ImageNet.
Two separate pipelines are defined: one for training images (which includes data augmentation)
and one for test images (which does not).

---

## Pipeline Steps

### 1. Color Space Conversion: BGR → RGB

OpenCV loads images in **BGR** channel order by default. Since ResNet18 was trained on
RGB images, the channel order must be corrected before any further processing:

```python
cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
```

This operation swaps the first and third channel. Skipping it would cause the network
to receive systematically incorrect color information, corrupting all downstream features.

---

### 2. Resize to 224×224

```python
T.Resize((224, 224))
```

All images are resized to 224×224 pixels using bilinear interpolation. This is the
spatial resolution expected by ResNet18 and ensures uniform input dimensions across
the entire dataset. After this step, every image has shape `(224, 224, 3)`.

---

### 3. Data Augmentation — Random Horizontal Flip (training only)

```python
T.RandomHorizontalFlip(p=0.5)
```

During training, each image is flipped horizontally with probability 0.5. This is a
standard **data augmentation** technique: it artificially increases the variety of
training samples by exposing the model to mirror-image variants of the same scene,
without altering the semantic content (a normal bottle remains a normal bottle when flipped).

This step is intentionally **omitted from the test pipeline**. Test images must faithfully
represent real-world inputs; applying random transformations at test time would introduce


In [ ]:
# ── ImageNet normalisation statistics ─────────────────────────────────────────
# ResNet18 was trained on ImageNet using these mean and standard deviation values
# for the three R, G, B channels. We must normalise our images the same way;
# otherwise the network "sees" value distributions different from what it learned on.
IMAGENET_MEAN = [0.485, 0.456, 0.406]   # per-channel means for R, G, B
IMAGENET_STD  = [0.229, 0.224, 0.225]   # per-channel standard deviations for R, G, B

# ── Transformation pipeline for TRAINING ──────────────────────────────────────
# T.Compose() takes a list of transforms and applies them sequentially,
# passing the output of one as the input to the next (like an assembly line)
train_transform = T.Compose([
    T.ToPILImage(),                           # NumPy array (H,W,3) -> PIL image (intermediate format)
    T.Resize((IMAGE_SIZE, IMAGE_SIZE)),        # resize every image to 224x224 pixels
    T.RandomHorizontalFlip(p=0.5),            # DATA AUGMENTATION: mirror the image horizontally
                                              # with 50% probability -> the network sees more variety
    T.ToTensor(),                             # PIL -> PyTorch tensor; scales values from [0,255] to [0.0,1.0]
                                              # and reorders axes from (H,W,C) to (C,H,W)
    T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),  # formula: x_norm = (x - mean) / std
])

# ── Transformation pipeline for TEST (no augmentation) ────────────────────────
# Test images must remain faithful to real inputs:
# we do not apply random flips (they would alter the image non-deterministically)
test_transform = T.Compose([
    T.ToPILImage(),
    T.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    T.ToTensor(),
    T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

# Print the list of transforms in the training pipeline
print("Train transform pipeline:")
for t in train_transform.transforms:
    print(f"  {t}")

In [ ]:
# ── Custom Dataset class ──────────────────────────────────────────────────────
# In PyTorch, to manage your own data you create a class that INHERITS from Dataset.
# "Inheriting" means our class gets all the functionality of Dataset
# and overrides / adds methods specific to our data.
class MVTecDataset(Dataset):
    """PyTorch Dataset for one MVTec AD category."""

    # __init__ is the "constructor": called when we create an instance of the class
    # (e.g. MVTecDataset(DATASET_ROOT, split="train", transform=train_transform))
    def __init__(self, root, split="train", transform=None):
        self.transform   = transform  # preprocessing pipeline to apply to each image
        self.image_paths = []  # list of paths to all images
        self.labels      = []  # 0 = normal, 1 = defective
        self.mask_paths  = []  # paths to ground-truth masks (None for normal images)

        if split == "train":
            # glob.glob() finds all .png files in the train/good folder
            for p in sorted(glob.glob(os.path.join(root, "train", "good", "*.png"))):
                self.image_paths.append(p)
                self.labels.append(0)        # all training images are normal
                self.mask_paths.append(None) # no mask for normal images
        else:
            # For the test set, iterate over all defect types (including "good")
            for defect_type in sorted(os.listdir(os.path.join(root, "test"))):
                img_dir  = os.path.join(root, "test", defect_type)
                mask_dir = os.path.join(root, "ground_truth", defect_type)
                # If the type is "good" -> label 0 (normal), otherwise 1 (defective)
                label    = 0 if defect_type == "good" else 1
                for p in sorted(glob.glob(os.path.join(img_dir, "*.png"))):
                    self.image_paths.append(p)
                    self.labels.append(label)
                    if label == 1:
                        # Build the path to the corresponding mask:
                        # os.path.basename() extracts just the file name from the full path
                        # .replace() substitutes one substring with another
                        mask_name = os.path.basename(p).replace(".png", "_mask.png")
                        self.mask_paths.append(os.path.join(mask_dir, mask_name))
                    else:
                        self.mask_paths.append(None)

    # __len__ tells PyTorch how many images are in the dataset.
    # DataLoader calls this internally to know when to stop iterating.
    def __len__(self):
        return len(self.image_paths)

    # __getitem__ retrieves a single sample by index (like list[i]).
    # PyTorch calls this method for every image it needs to load.
    def __getitem__(self, idx):
        # Read the image and convert from BGR to RGB
        img = cv2.cvtColor(cv2.imread(self.image_paths[idx]), cv2.COLOR_BGR2RGB)
        # Apply the transformation pipeline if provided, otherwise just convert to tensor
        img = self.transform(img) if self.transform else T.ToTensor()(img)

        # ── Load the ground-truth mask ─────────────────────────────────────────
        mask_path = self.mask_paths[idx]
        if mask_path and os.path.exists(mask_path):
            # IMREAD_GRAYSCALE reads the image as a single-channel grey-scale array
            mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
            # Resize the mask to match the image dimensions
            mask = cv2.resize(mask, (IMAGE_SIZE, IMAGE_SIZE))
            # Convert to a binary mask: pixel > 0 -> 1 (defect), pixel = 0 -> 0 (normal)
            mask = (mask > 0).astype(np.uint8)
        else:
            # For normal images: all-zero mask (no defect)
            mask = np.zeros((IMAGE_SIZE, IMAGE_SIZE), dtype=np.uint8)

        # Return a tuple (image, label, mask)
        return img, self.labels[idx], mask

In [ ]:
# ── Instantiate the datasets ──────────────────────────────────────────────────
# We create two MVTecDataset objects: one for training, one for testing
train_dataset = MVTecDataset(DATASET_ROOT, split="train", transform=train_transform)
test_dataset  = MVTecDataset(DATASET_ROOT, split="test",  transform=test_transform)

# ── Create the DataLoaders ────────────────────────────────────────────────────
# DataLoader wraps a Dataset and handles:
#   - loading data in batches of size batch_size
#   - optional random shuffling (shuffle=True)
#   - parallel loading with multiple worker processes (num_workers)
# batch_size=32: process 32 images at a time (balance between speed and GPU memory)
# shuffle=False: keep the original order (useful to align predictions with labels later)
# num_workers=2: 2 background processes prepare batches while the GPU processes the current one
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=False, num_workers=2)
test_loader  = DataLoader(test_dataset,  batch_size=32, shuffle=False, num_workers=2)

print(f"Train : {len(train_dataset)} images")
# sum() on a list of 0s and 1s counts how many equal 1 (the defective images)
print(f"Test  : {len(test_dataset)} images  ({sum(test_dataset.labels)} defective)")

# ── Verify tensor shapes ──────────────────────────────────────────────────────
# iter() creates an iterator from the DataLoader; next() pulls the first element (first batch)
imgs, labels, masks = next(iter(train_loader))
print(f"\nBatch shapes:")
# .shape returns the tensor dimensions as a torch.Size object
# list() converts it to a plain list for easier reading
# Image shape: (batch_size=32, channels=3, height=224, width=224)
print(f"  Images : {list(imgs.shape)}   → (batch, channels, H, W)")
print(f"  Labels : {list(labels.shape)}")
print(f"  Masks  : {list(masks.shape)}")

# Section 3 — Feature Extraction with ResNet18

## Motivation

Comparing images directly at the pixel level is not a viable strategy for anomaly detection.
A 224×224 RGB image contains 150,528 numerical values, and even minor variations in
lighting, position, or camera angle cause large pixel-level differences between otherwise
identical images. What is needed is a more abstract representation — a compact numerical
descriptor that captures *what* is present in an image, independently of low-level
pixel-level details.

This compact representation is called a **feature vector**, and the process of computing
it from raw image data is called **feature extraction**.

---

## Convolutional Neural Networks

A Convolutional Neural Network (CNN) is a sequence of layers, each of which transforms
its input into a progressively more abstract representation. The core operation is the
**convolution**: a small filter (e.g., 3×3 pixels) is slid across the entire spatial
extent of the input. At each position, the filter values are multiplied element-wise with
the corresponding input values and summed, producing a single scalar. The result is a
feature map that responds strongly where the input matches the filter's pattern.

Crucially, filter values are not hand-designed — they are **learned from data** during
training by minimizing a task-specific loss function. A network trained on large-scale
image classification therefore learns, in its early and intermediate layers, general-purpose
visual detectors: edges, textures, local patterns, and structural elements that are
universally useful for describing visual content.

---

## ResNet18 Architecture

ResNet18 is a CNN architecture introduced by He et al. (2015). The number 18 refers to
the count of layers with learnable parameters. For a 224×224 RGB input, the spatial
resolution and channel depth evolve through the network as follows:

| Stage | Output shape | Typical content |
|-------|-------------|-----------------|
| Input | (3, 224, 224) | Raw RGB pixels |
| conv1 + pool | (64, 56, 56) | Edges, intensity gradients |
| layer1 | (64, 56, 56) | Simple textures, corners |
| layer2 | (128, 28, 28) | Local patterns, repetitive structures |
| layer3 | (256, 14, 14) | Object parts, complex shapes |
| layer4 | (512, 7, 7) | High-level semantic concepts |
| fc | (1000,) | ImageNet class scores |

As depth increases, spatial resolution decreases and the number of channels grows.
Deeper layers capture increasingly abstract and class-specific information.

---

## Why Truncate at Layer2?

The network is truncated after `layer2`, discarding all subsequent layers including
the classification head. This design choice is motivated by two considerations:

**Spatial resolution.** Layer2 produces a 28×28 feature map, meaning the input image
is described at 784 distinct spatial locations. This granularity is sufficient for
localizing defects at a meaningful scale. Deeper layers (14×14, 7×7) would lose too
much spatial detail.

**Semantic level.** Layer2 features capture mid-level visual patterns — textures and
local structures — which are precisely the information that distinguishes a normal surface
from a defective one (a scratch, a fracture, a contamination spot). Features from
deeper layers are optimized for discriminating between semantic categories
(e.g., "bottle" vs. "cat"), not for detecting subtle surface irregularities.

---

## Spatial Patch Descriptors

The output of layer2 for a single image is a tensor of shape `(128, 28, 28)`.
This can be interpreted as a 28×28 grid of **patch descriptors**: each of the
784 spatial positions corresponds to a receptive field of approximately 8×8 pixels
in the original image, and is described by a 128-dimensional feature vector encoding
the local appearance of that region.

```
One image → 784 patch descriptors × 128 features each
```

This patch-level representation is the key to spatial anomaly localization.
Rather than producing a single score per image, the pipeline produces a score per
spatial position, enabling the construction of a pixel-level anomaly heatmap.

---

## Transfer Learning

ResNet18 is used with weights pre-trained on ImageNet (1.2M images, 1000 classes),
without any fine-tuning on the MVTec dataset. The network is set to evaluation mode
(`model.eval()`) and gradient computation is disabled (`torch.no_grad()`), since no
parameter updates are performed.

This approach is justified by the generalization properties of CNN features: early and
intermediate layers learn visual primitives (edges, textures, local patterns) that are
not specific to any particular object category and transfer effectively across domains.
A network that has processed 1.2 million diverse images has developed rich, general-purpose
visual representations that remain highly discriminative even when applied to an
unseen industrial domain.

---

## Implementation

### Feature Extractor

```python
class FeatureExtractor(nn.Module):
 """ResNet18 truncated after layer2. Outputs (B, 128, 28, 28) for 224×224 input."""

 def __init__(self):
 super().__init__()
 backbone = resnet18(weights=ResNet18_Weights.IMAGENET1K_V1)
 self.features = nn.Sequential(
 backbone.conv1, backbone.bn1, backbone.relu, backbone.maxpool,
 backbone.layer1, backbone.layer2,
 )

 def forward(self, x):
 return self.features(x)
```

### Feature Extraction Loop

```python
@torch.no_grad()
def extract_features(loader, model, device):
 all_feats = []
 for imgs, _, _ in tqdm(loader, desc="Extracting"):
 feats = model(imgs.to(device)) # (B, 128, 28, 28)
 all_feats.append(feats.cpu())
 return torch.cat(all_feats, dim=0)
```

The `@torch.no_grad()` decorator disables gradient tracking for the duration of the
function. Since no backpropagation is performed, this reduces memory consumption and
accelerates inference. Features are moved back to CPU after each batch to avoid
exhausting GPU memory.

For the full training set of ~209 images, the output tensor has shape `(209, 128, 28, 28)`,
representing 209 × 784 = 163,856 patch descriptors of dimension 128.

---

## PCA Visualization

To verify that the extracted features carry meaningful discriminative information,
a 2D PCA projection is computed from the globally average-pooled features
(shape: `(N, 128)`) of both training and test images.

If the learned representation is effective, normal training images and normal test images
should form a compact, well-defined cluster, while defective test images should appear
as outliers at the periphery of that cluster. This spatial separation confirms that
distance-based anomaly scoring — the approach used in Section 4 — is a sound strategy
for this feature space.

In [ ]:
# ── Feature extractor definition ──────────────────────────────────────────────
# We create a class that inherits from nn.Module (the base class for all PyTorch networks)
class FeatureExtractor(nn.Module):
    """ResNet18 truncated after layer2. Outputs (B, 128, 28, 28) for 224x224 input."""

    def __init__(self):
        # super().__init__() calls the parent class constructor (nn.Module).
        # This is mandatory before defining any layers.
        super().__init__()

        # Load ResNet18 with weights pre-trained on ImageNet.
        # weights=ResNet18_Weights.IMAGENET1K_V1 downloads the weights automatically if not cached.
        backbone = resnet18(weights=ResNet18_Weights.IMAGENET1K_V1)

        # nn.Sequential() chains modules in series: the output of one becomes the input of the next.
        # We take only the first layers of ResNet18 up to layer2,
        # discarding layer3, layer4, and the final classifier (fc).
        # This way we "cut" the network and keep only the part that extracts local features.
        self.features = nn.Sequential(
            backbone.conv1,    # first convolutional layer: detects edges
            backbone.bn1,      # batch normalisation: stabilises values between layers
            backbone.relu,     # ReLU activation function: introduces non-linearity
            backbone.maxpool,  # reduces spatial resolution by 2x (max pooling)
            backbone.layer1,   # residual block 1: low-level features (simple textures)
            backbone.layer2,   # residual block 2: mid-level features (complex edges, patterns)
            # layer3, layer4 (too abstract) and fc (final classifier) are NOT used
        )

    # forward() defines what happens when we pass an input to the network (the "forward pass").
    # It is called automatically when we write model(input).
    def forward(self, x):
        return self.features(x)


# ── Instantiate and verify ────────────────────────────────────────────────────
# .to(DEVICE) moves the network to the GPU (if available) or CPU
# .eval() switches the network to evaluation mode: disables dropout and stochastic batch norm
model = FeatureExtractor().to(DEVICE).eval()

# Verify input/output shapes with a dummy all-zero tensor
# torch.no_grad() disables gradient computation (not needed during inference, saves memory)
with torch.no_grad():
    # Create a zero tensor of shape (1, 3, 224, 224) = 1 image, 3 channels, 224x224 pixels
    dummy = torch.zeros(1, 3, IMAGE_SIZE, IMAGE_SIZE).to(DEVICE)
    out   = model(dummy)

print(f"Input  : {list(dummy.shape)}")
# Expected output: (1, 128, 28, 28) = 1 image, 128 channels, 28x28 spatial map
print(f"Output : {list(out.shape)}")
# 28*28 = 784: each image produces 784 patch descriptors, each of dimension 128
print(f"\nEach image → {28 * 28} spatial patches × 128 features")


In [ ]:


# ── Decorator to disable gradient computation ─────────────────────────────────
# @torch.no_grad() is a "decorator": it modifies the behaviour of the function below it.
# Here it tells PyTorch not to compute gradients during execution.
# Gradients are only needed for training (backpropagation); since we are not training,
# disabling them saves memory and speeds up the computation.
@torch.no_grad()
def extract_features(loader, model, device):
    """Returns feature maps (N, 128, 28, 28) for all images in a DataLoader."""
    all_feats = []  # empty list where we accumulate features from each batch

    # The for loop iterates over all batches in the DataLoader
    # tqdm() wraps the iterable and displays a progress bar
    # desc= sets the descriptive text next to the bar
    for imgs, _, _ in tqdm(loader, desc="Extracting"):
        # _ (underscore) is a "throwaway" variable (labels and masks are not needed here)
        # imgs.to(device) moves the batch to the GPU/CPU
        feats = model(imgs.to(device))   # forward pass: (B, 3, 224, 224) -> (B, 128, 28, 28)
        all_feats.append(feats.cpu())    # .cpu() moves the tensor back to RAM (from GPU VRAM)

    # torch.cat() concatenates all tensors in the list along dimension 0 (images)
    # Result: (N_total, 128, 28, 28) where N_total = total number of images
    return torch.cat(all_feats, dim=0)


print("Extracting training features...")
train_features = extract_features(train_loader, model, DEVICE)

print("Extracting test features...")
test_features  = extract_features(test_loader,  model, DEVICE)

print(f"\nTrain features : {train_features.shape}")
print(f"Test  features : {test_features.shape}")


In [ ]:
# ── Dimensionality reduction for feature visualisation ────────────────────────
# Feature maps have shape (N, 128, 28, 28). To visualise them in 2D we use PCA.
# First we "collapse" the spatial dimensions (28x28) with Global Average Pooling:
# we average over all spatial positions -> each image becomes a 128-dimensional vector.
# .mean(dim=[2, 3]) computes the mean over dimensions 2 and 3 (height and width)
# .numpy() converts the PyTorch tensor to a NumPy array (required by scikit-learn)
train_pooled = train_features.mean(dim=[2, 3]).numpy()  # (N_train, 128)
test_pooled  = test_features.mean(dim=[2, 3]).numpy()   # (N_test, 128)
test_labels  = np.array(test_dataset.labels)            # labels: 0=normal, 1=defective

# ── PCA: from 128 dimensions down to 2 ───────────────────────────────────────
# PCA(n_components=2) finds the 2 directions of maximum variance in the 128-dim space.
# .fit() computes the principal components on the training data.
# .transform() projects the data onto the 2 new dimensions.
pca = PCA(n_components=2).fit(train_pooled)
train_2d = pca.transform(train_pooled)  # (N_train, 2): 2D coordinates for each training image
test_2d  = pca.transform(test_pooled)   # (N_test, 2): 2D coordinates for each test image

# ── Scatter plot visualisation ────────────────────────────────────────────────
plt.figure(figsize=(8, 6))
# Blue dots: training images (all normal)
plt.scatter(train_2d[:, 0], train_2d[:, 1],
            c="steelblue", alpha=0.4, label="Train (normal)")
# Green triangles: normal test images (test_labels == 0)
# Boolean indexing: test_2d[mask, :] selects only rows where mask is True
plt.scatter(test_2d[test_labels == 0, 0], test_2d[test_labels == 0, 1],
            c="green", alpha=0.6, marker="^", label="Test — normal")
# Red crosses: defective test images (test_labels == 1)
plt.scatter(test_2d[test_labels == 1, 0], test_2d[test_labels == 1, 1],
            c="tomato", alpha=0.7, marker="x", s=80, label="Test — defective")
plt.xlabel("PC 1")   # first principal component
plt.ylabel("PC 2")   # second principal component
plt.title("PCA of ResNet18 Feature Vectors (global avg pooling)")
plt.legend()
plt.grid(True, alpha=0.3)  # semi-transparent background grid
plt.tight_layout()
plt.show()
# explained_variance_ratio_ shows how much total variance the chosen components capture
print(f"Variance explained by PC1+PC2: {pca.explained_variance_ratio_.sum():.1%}")

# Section 4 — Anomaly Scoring with k-Nearest Neighbors

## Overview

At this stage, every training image has been converted into 784 patch descriptors of
dimension 128 by the ResNet18 backbone. The goal of this section is to use these
descriptors to assign an anomaly score to each test image — a single number that
measures how unusual the image looks compared to the normal training distribution.

The approach is entirely distance-based: patches from normal images cluster tightly
in feature space, while patches from defective regions fall far from that cluster.
Distance to the nearest normal patches is therefore a direct and interpretable
anomaly signal.

---

## The Memory Bank

All patch descriptors extracted from all normal training images are collected into a
single structure called the **memory bank**. This is a complete reference catalogue
of every local appearance that a normal bottle can exhibit.

```
209 images × 784 patches = 163,856 patch descriptors
```

Each descriptor is a 128-dimensional vector. The memory bank is therefore a matrix
of shape `(163,856 × 128)`.

In code, this is obtained by flattening the spatial dimensions of the training feature tensor:

```python
memory_bank = train_features.permute(0, 2, 3, 1).reshape(-1, C).numpy()
```

### Subsampling

163,856 vectors require significant memory and make k-NN queries slow. The memory bank
is therefore randomly subsampled to 50,000 vectors:

```python
MAX_PATCHES = 50_000
idx = np.random.choice(len(memory_bank), MAX_PATCHES, replace=False)
memory_bank = memory_bank[idx]
```

This is a practical approximation with negligible impact on performance: normal bottles
are visually very similar to each other, so 50,000 vectors already provide dense coverage
of the normal feature space.

---

## k-NN Anomaly Scoring

### Fitting the Model

A `NearestNeighbors` model is fitted on the memory bank:

```python
knn = NearestNeighbors(n_neighbors=9, metric="euclidean", algorithm="ball_tree")
knn.fit(memory_bank)
```

No learning takes place here in the traditional sense — the model simply stores the
memory bank vectors and builds an index that allows fast nearest-neighbor queries.

### Patch-level Scoring

For each test image, the 784 patch descriptors are extracted and scored individually.
For each patch descriptor:

1. The 9 nearest neighbors in the memory bank are retrieved
2. The mean Euclidean distance to those 9 neighbors is computed
3. This mean distance is the **anomaly score** of that patch
```python
dists, _ = knn.kneighbors(patches[i]) # (784, 9)
patch_score = dists.mean(axis=1) # (784,)
```

### Why Distance Measures Anomaly

A patch from a normal region will find many close neighbors in the memory bank —
because the memory bank is dense with normal appearances. Its mean distance will be low.

A patch from a defective region (a scratch, a fracture, a contamination spot) will
find no close neighbors — its appearance has never been seen during training.
Its mean distance will be high.

```
Low score → patch resembles known normal regions → likely normal
High score → patch far from all normal regions → likely defective
```

### From Patch Scores to Image Score

Each test image produces 784 patch scores, arranged in a spatial 28×28 map
called the **patch map**:

```python
spatial_map = patch_score.reshape(H, W) # (28, 28)
```

These 784 scores are aggregated into a single image-level anomaly score by taking
the **maximum**:

```python
image_score = spatial_map.max()
```

The maximum is used — rather than the mean — because defects are spatially localized
and affect only a small fraction of patches. Averaging would dilute the anomaly signal
across the many normal patches surrounding the defect. The maximum ensures that even
a single highly anomalous patch is sufficient to flag the entire image.

---

## Score Distribution

The histogram of image-level anomaly scores shows the separation between normal and
defective test images. A well-functioning system produces two clearly distinct
distributions: normal images concentrated at low scores, defective images at high scores.
The degree of overlap between the two distributions directly determines the
classification performance measured in Section 6.

---

## Full Pipeline Summary

```
Training
────────────────────────────────────────────
~209 normal images
 │
 ▼ ResNet18 (layer2)
163,856 patch descriptors (128-dim each)
 │
 ▼ random subsampling
50,000 vectors → memory bank
 │
 ▼ knn.fit()
k-NN index ready

Test (per image)
────────────────────────────────────────────
1 new image
 │
 ▼ ResNet18 (layer2)
784 patch descriptors
 │
 ▼ knn.kneighbors()
784 mean distances → patch map (28×28)
 │
 ▼ max()
1 anomaly score → normal or defective?
```

In [ ]:
# ── Build the Memory Bank ─────────────────────────────────────────────────────
# The memory bank is the "database" of all normal patches seen during training.
# Each training image produces 28x28 = 784 patch descriptors of dimension 128.
# We collect all these descriptors into a 2D matrix (rows = patches, columns = features).

# train_features has shape (N_train, 128, 28, 28)
N, C, H, W  = train_features.shape  # unpack the 4 dimensions into separate variables

# .permute(0, 2, 3, 1) reorders dimensions: (N, 128, 28, 28) -> (N, 28, 28, 128)
# .reshape(-1, C) flattens the first 3 dimensions into one:
#   (N, 28, 28, 128) -> (N*28*28, 128) = (N*784, 128)
#   -1 means "infer this dimension automatically"
# .numpy() converts to a NumPy array for scikit-learn
memory_bank = train_features.permute(0, 2, 3, 1).reshape(-1, C).numpy()
print(f"Full memory bank : {memory_bank.shape}")

# ── Sub-sample to keep memory manageable on free Colab ───────────────────────
# With 209 training images: 209 * 784 = ~164,000 patches -> could exhaust Colab RAM
# We draw a random subset of MAX_PATCHES patches
MAX_PATCHES = 50_000  # the underscore is just a readability separator (= 50000)
if len(memory_bank) > MAX_PATCHES:
    # np.random.choice() picks MAX_PATCHES random indices without replacement
    idx = np.random.choice(len(memory_bank), MAX_PATCHES, replace=False)
    memory_bank = memory_bank[idx]   # select only the rows at the chosen indices
    print(f"Subsampled to    : {memory_bank.shape}")

# ── Fit the k-NN ──────────────────────────────────────────────────────────────
# NearestNeighbors is scikit-learn's k-NN algorithm: the same API you already know!
# n_neighbors=9: for each test patch, find its 9 nearest neighbours
# metric="euclidean": use Euclidean distance (square root of sum of squared differences)
# algorithm="ball_tree": efficient data structure for nearest-neighbour search in high dimensions
# n_jobs=-1: use all available CPU cores to parallelise the search
knn = NearestNeighbors(n_neighbors=9, metric="euclidean",
                       algorithm="ball_tree", n_jobs=-1)
knn.fit(memory_bank)  # .fit() stores all normal patches (no "training" in the classical sense)
print("k-NN fitted on memory bank.")

In [ ]:
def compute_scores(features, knn):
    """
    Returns:
      image_scores (N,)      — max patch score per image
      patch_maps   (N, H, W) — spatial anomaly map per image
    """
    # Unpack the feature tensor dimensions
    N, C, H, W = features.shape  # N=num images, C=128 channels, H=W=28 (spatial map)

    # Reshape: (N, C, H, W) -> (N, H*W, C) = (N, 784, 128)
    # Each image becomes a matrix of 784 rows (patches) x 128 columns (features)
    patches = features.permute(0, 2, 3, 1).reshape(N, H * W, C).numpy()

    image_scores = []  # one anomaly score per image
    patch_maps   = []  # one 28x28 spatial map per image

    # Iterate over each image individually (range(N) generates [0, 1, 2, ..., N-1])
    for i in tqdm(range(N), desc="Scoring"):
        # knn.kneighbors(patches[i]) finds the 9 nearest neighbours for each of the 784 patches
        # patches[i] has shape (784, 128): 784 query vectors, each of dimension 128
        # dists has shape (784, 9): for each patch, the distances to its 9 nearest neighbours
        # _ (neighbour indices) is not needed, we only use the distances
        dists, _    = knn.kneighbors(patches[i])  # (784, 9)

        # Per-patch score = mean distance to the k neighbours
        # .mean(axis=1) averages across columns (the 9 neighbours) -> (784,)
        patch_score = dists.mean(axis=1)

        # Reshape the 784 scores into a 28x28 spatial map
        spatial_map = patch_score.reshape(H, W)

        # Image score = maximum value across all patches
        # (a single anomalous patch is enough to flag the whole image as defective)
        image_scores.append(spatial_map.max())
        patch_maps.append(spatial_map)

    # np.array() converts the list of scalars to a NumPy array of shape (N,)
    # np.stack() stacks the N maps of shape (28, 28) into an array of shape (N, 28, 28)
    return np.array(image_scores), np.stack(patch_maps)


image_scores, patch_maps = compute_scores(test_features, knn)
print(f"Image scores : {image_scores.shape}")
print(f"Patch maps   : {patch_maps.shape}")

In [ ]:
# ── Split scores by category ──────────────────────────────────────────────────
# NumPy "boolean indexing": array[mask] selects only elements where the mask is True
normal_scores  = image_scores[test_labels == 0]   # scores for normal images
anomaly_scores = image_scores[test_labels == 1]   # scores for defective images

# ── Histogram of score distributions ──────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 4))

# ax.hist() draws a histogram: groups values into "bins" and counts how many fall in each
# bins=30: divide the value range into 30 intervals
# alpha=0.65: opacity (0=transparent, 1=opaque); useful to see the overlap between distributions
# edgecolor: colour of each bar's border
ax.hist(normal_scores,  bins=30, alpha=0.65, label="Normal",
        color="steelblue", edgecolor="white")
ax.hist(anomaly_scores, bins=30, alpha=0.65, label="Defective",
        color="tomato",   edgecolor="white")

ax.set_xlabel("Anomaly Score")
ax.set_ylabel("Count")
ax.set_title("Distribution of Image-level Anomaly Scores")
ax.legend()  # display the legend (links colours to labels)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# .mean() and .std() compute the mean and standard deviation of the array
# :.3f formats the number with 3 decimal places
print(f"Normal    — mean: {normal_scores.mean():.3f}  std: {normal_scores.std():.3f}")
print(f"Defective — mean: {anomaly_scores.mean():.3f}  std: {anomaly_scores.std():.3f}")

# Section 5 — Post-processing: Pixel-level Anomaly Maps

## Overview

The k-NN scoring stage produces a **patch map** of shape `(28×28)` for each test image —
a spatial grid where each cell contains the anomaly score of the corresponding image region.
This resolution is too coarse for pixel-level defect localization: each cell covers
approximately 8×8 pixels of the original 224×224 image.

The post-processing pipeline transforms this raw 28×28 score map into a clean,
pixel-level binary mask through four sequential steps: Gaussian smoothing, bilinear
upsampling, percentile thresholding, and morphological refinement.

---

## Step 1 — Gaussian Smoothing

```python
smoothed = gaussian_filter(pm, sigma=4)
```

The raw patch map contains abrupt transitions at patch boundaries — adjacent cells
may have very different scores even if they belong to the same defective region,
simply because the patch grid is discrete.

A Gaussian filter smooths these transitions by replacing each value with a weighted
average of its neighbors, where the weights follow a Gaussian (bell-shaped) distribution.
The parameter `sigma=4` controls the smoothing radius: larger values produce smoother
maps at the cost of spatial precision.

This step reduces block artifacts and produces a more spatially coherent heatmap
where anomaly scores spread gradually across neighboring regions.

---

## Step 2 — Bilinear Upsampling

```python
upsampled = cv2.resize(smoothed, (224, 224), interpolation=cv2.INTER_LINEAR)
```

The smoothed 28×28 map is upsampled to 224×224 using bilinear interpolation —
the same resolution as the original input image. Each output pixel is computed as
a weighted average of the four nearest input values, producing a smooth continuous
heatmap at full image resolution.

After this step, every pixel in the original image has an associated anomaly score.

---

## Step 3 — Normalization and Percentile Thresholding

```python
norm_map = (anomaly_map - vmin) / (vmax - vmin + 1e-8)
thresh = np.percentile(norm_map, percentile=95)
binary = (norm_map >= thresh).astype(np.uint8)
```

The upsampled map is first normalized to [0, 1] by subtracting the minimum and dividing
by the range. This makes the threshold consistent across images regardless of the
absolute scale of the anomaly scores.

A percentile-based threshold is then applied: the top 5% of pixel values (those above
the 95th percentile) are flagged as anomalous. This adaptive strategy avoids the need
to manually set a fixed threshold and ensures that every image produces a non-empty
prediction — which is appropriate for evaluation purposes.

The result is a binary mask where 1 indicates a predicted defective pixel and 0
indicates a predicted normal pixel.

---

## Step 4 — Morphological Refinement

```python
kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))
closed = cv2.morphologyEx(binary, cv2.MORPH_CLOSE, kernel)
cleaned = cv2.morphologyEx(closed, cv2.MORPH_OPEN, kernel)
```

Two classical morphological operations are applied sequentially to refine the binary mask:

**Morphological Closing** (dilation followed by erosion) fills small holes and gaps
inside detected defective regions. If the anomaly map correctly identifies the outline
of a defect but leaves isolated normal pixels inside, closing reconnects these regions
into a solid contiguous area.

**Morphological Opening** (erosion followed by dilation) removes small isolated clusters
of predicted defective pixels that are surrounded by normal regions. These are typically
noise artifacts produced by the thresholding step rather than true defects.

Both operations use an elliptical structuring element of size 5×5 pixels, which
provides isotropic behavior — the operations are not biased toward horizontal or
vertical directions.

---

## Output

The post-processing pipeline produces two outputs per test image:

- `norm_map` — the continuous normalized anomaly heatmap at 224×224 resolution,
 used for pixel-level AUROC evaluation.
- `pred_mask` — the binary defect segmentation mask at 224×224 resolution,
 used for IoU and Dice evaluation against the ground truth masks.
---

## Summary

| Step | Input | Output | Purpose |
|------|-------|--------|---------|
| Gaussian smoothing | (28, 28) raw scores | (28, 28) smooth scores | Remove patch boundary artifacts |
| Bilinear upsampling | (28, 28) | (224, 224) | Full image resolution |
| Normalization + threshold | (224, 224) continuous | (224, 224) binary | Separate defect from background |
| Morphological ops | (224, 224) noisy binary | (224, 224) clean binary | Fill holes, remove noise |

In [ ]:
def build_anomaly_maps(patch_maps, target_size=IMAGE_SIZE, sigma=4):
    """Gaussian smooth + bilinear upsample (N, 28, 28) -> (N, target_size, target_size)."""
    maps = []

    # Iterate over each 28x28 map (one per test image)
    for pm in patch_maps:

        # STEP 1 — Gaussian smoothing
        # gaussian_filter() blurs the map by applying a Gaussian filter.
        # sigma controls the "width" of the blur: larger values -> more blurring.
        # This removes blocky artefacts at the boundaries between adjacent patches.
        smoothed  = gaussian_filter(pm, sigma=sigma)

        # STEP 2 — Bilinear upsampling
        # cv2.resize() rescales the image; INTER_LINEAR uses bilinear interpolation:
        # new pixel values are computed as a weighted average of neighbouring pixels.
        # We scale the map from 28x28 up to 224x224 (the original image resolution).
        upsampled = cv2.resize(smoothed, (target_size, target_size),
                               interpolation=cv2.INTER_LINEAR)
        maps.append(upsampled)

    # np.stack() stacks the maps into a single array of shape (N, 224, 224)
    return np.stack(maps)


anomaly_maps = build_anomaly_maps(patch_maps)
print(f"Anomaly maps: {anomaly_maps.shape}")

In [ ]:
def make_binary_mask(anomaly_map, percentile=95):
    """Normalize -> threshold -> morphological cleaning. Returns (norm_map, binary_mask)."""

    # STEP 1 — Min-max normalisation: scale values to the interval [0, 1]
    # Formula: (x - min) / (max - min)
    # 1e-8 is a small value added to the denominator to avoid division by zero
    # (if min == max, all pixels would have the same score)
    vmin, vmax = anomaly_map.min(), anomaly_map.max()
    norm_map   = (anomaly_map - vmin) / (vmax - vmin + 1e-8)

    # STEP 2 — Percentile-based thresholding
    # np.percentile(array, 95) finds the value below which 95% of pixels fall
    # -> the top 5% of pixels are considered "anomalous"
    thresh = np.percentile(norm_map, percentile)
    # Create a binary mask: 1 where the pixel is above the threshold, 0 elsewhere
    # .astype(np.uint8) converts booleans to integers (0/1) as required by OpenCV
    binary = (norm_map >= thresh).astype(np.uint8)

    # STEP 3 — Morphological operations to clean up the mask
    # cv2.getStructuringElement() creates a "kernel" (structuring element) shaped as an ellipse
    # of size 5x5. The kernel is the "tool" used by morphological operations.
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))

    # MORPH_CLOSE = dilation followed by erosion:
    # fills small holes inside white regions (detected defects)
    closed  = cv2.morphologyEx(binary, cv2.MORPH_CLOSE, kernel)

    # MORPH_OPEN = erosion followed by dilation:
    # removes small isolated white dots (noise) that are not part of real defects
    cleaned = cv2.morphologyEx(closed, cv2.MORPH_OPEN,  kernel)

    return norm_map, cleaned   # return both the continuous map and the binary mask


# ── Apply to all test images ──────────────────────────────────────────────────
# Collect ground-truth masks for every image in the test set
# test_dataset[i] returns the tuple (img, label, mask); [2] picks only the mask
test_masks_gt  = np.stack([test_dataset[i][2] for i in range(len(test_dataset))])
pred_norm_maps = []   # normalised anomaly maps (continuous values in [0, 1])
pred_masks     = []   # predicted binary masks (values 0 or 1)

for amap in anomaly_maps:
    nmap, mask = make_binary_mask(amap, percentile=95)
    pred_norm_maps.append(nmap)
    pred_masks.append(mask)

# Convert lists to NumPy arrays for subsequent vectorised operations
pred_norm_maps = np.stack(pred_norm_maps)
pred_masks     = np.stack(pred_masks)
print(f"Predicted masks: {pred_masks.shape}")

In [ ]:
# ── De-normalisation helper to display original images ────────────────────────
# Images in the dataset are normalised (mean subtracted, divided by std).
# To display them correctly we must invert that operation.
def denormalize(tensor):
    # Create mean and std tensors with shape (3, 1, 1) to leverage broadcasting:
    # PyTorch allows operations between tensors of different shapes when dimensions are compatible.
    # (3, 1, 1) can operate with (3, H, W) by applying the values to every pixel.
    mean = torch.tensor(IMAGENET_MEAN).view(3, 1, 1)
    std  = torch.tensor(IMAGENET_STD).view(3, 1, 1)

    # Invert normalisation: x_orig = x_norm * std + mean
    # .clamp(0, 1) clips values outside [0, 1] (fixes minor numerical errors)
    # .permute(1, 2, 0) reorders from (C, H, W) to (H, W, C) as required by matplotlib
    # .numpy() converts to a NumPy array
    return (tensor * std + mean).clamp(0, 1).permute(1, 2, 0).numpy()


# Take the first 4 indices of defective images in the test set
# np.where() returns a tuple of arrays with indices where the condition is True
# [0] picks the first (and only) array from the tuple; [:4] takes the first 4 elements
defect_indices = np.where(test_labels == 1)[0][:4]

# ── 4x4 visualisation grid ────────────────────────────────────────────────────
# 4 rows (one per defective image) x 4 columns (original, GT mask, heatmap, predicted mask)
fig, axes = plt.subplots(4, 4, figsize=(14, 14))

# Column headers (only in the first row)
for col, title in enumerate(["Original", "GT Mask", "Anomaly Heatmap", "Predicted Mask"]):
    axes[0, col].set_title(title, fontsize=12, fontweight="bold")

# enumerate() adds a counter (row=0,1,2,3) to the elements of the list
for row, idx in enumerate(defect_indices):
    img_t, label, gt_mask = test_dataset[idx]   # retrieve the sample

    axes[row, 0].imshow(denormalize(img_t))             # original image (de-normalised)
    axes[row, 1].imshow(gt_mask, cmap="Reds", vmin=0, vmax=1)         # ground-truth mask
    axes[row, 2].imshow(pred_norm_maps[idx], cmap="hot")               # anomaly heatmap
    axes[row, 3].imshow(pred_masks[idx], cmap="Reds", vmin=0, vmax=1) # predicted binary mask

    # Hide axes for all panels in the current row
    for ax in axes[row]:
        ax.axis("off")

plt.tight_layout()
plt.show()

# Section 6 — Evaluation & Results

## Overview

The system is evaluated at two levels of granularity:

- **Image-level** — can the model correctly distinguish normal images from defective ones?
- **Pixel-level** — does the predicted defect mask spatially overlap with the ground truth mask?
Each level uses metrics appropriate to its task.

---

## Image-level Evaluation: AUROC

### Definition

The **Area Under the ROC Curve (AUROC)** measures how well the image-level anomaly
score separates normal images from defective ones, independently of any threshold choice.

The ROC curve is constructed by sweeping a threshold across all possible values and
recording, at each threshold:

- **True Positive Rate (TPR / Recall)** — the fraction of defective images correctly
 flagged as anomalous.
- **False Positive Rate (FPR)** — the fraction of normal images incorrectly flagged
 as anomalous.
The AUROC is the area under this curve:

- **1.0** — perfect separation: every defective image scores higher than every normal image.
- **0.5** — random baseline: the score carries no discriminative information.
- **< 0.5** — the score is inversely correlated with anomaly (worse than random).
### Why AUROC and Not Accuracy

Accuracy requires a fixed threshold and is sensitive to class imbalance. The test set
contains more defective images than normal ones, making accuracy an unreliable metric.
AUROC evaluates the ranking quality of the scores across all thresholds simultaneously,
providing a more robust and threshold-independent measure of performance.

---

## Pixel-level Evaluation: IoU and Dice

Pixel-level metrics compare the predicted binary mask to the ground truth mask provided
by the MVTec dataset, pixel by pixel. Only defective test images are included in this
evaluation — normal images have empty ground truth masks by definition.

### Intersection over Union (IoU)

Also known as the Jaccard Index:

```
IoU = |Pred ∩ GT| / |Pred ∪ GT|
```

IoU measures the overlap between the predicted mask and the ground truth mask relative
to their union. It ranges from 0 (no overlap) to 1 (perfect match) and penalizes
both missed defect pixels (false negatives) and incorrectly flagged pixels (false positives).

### Dice Coefficient

```
Dice = 2 × |Pred ∩ GT| / (|Pred| + |GT|)
```

The Dice coefficient is equivalent to the F1-score computed at the pixel level.
It ranges from 0 to 1 and weights precision and recall equally. Compared to IoU,
Dice is less sensitive to over-segmentation because it normalizes by the sum of
mask sizes rather than their union.

### Relationship Between IoU and Dice

The two metrics are monotonically related — if one improves, the other does too.
They differ in how strongly they penalize errors:

```
Dice = 2 × IoU / (1 + IoU)
```

For this reason, Dice values are always higher than the corresponding IoU values
for the same prediction.

---

## Results

| Metric | Value |
|--------|-------|
| Image-level AUROC | *reported after execution* |
| Pixel-level IoU | *reported after execution* |
| Pixel-level Dice | *reported after execution* |

---

## ROC Curve

The ROC curve plots TPR against FPR at every possible threshold. The shaded area
under the curve corresponds to the AUROC value. A curve that hugs the top-left corner
indicates strong performance — high recall at low false alarm rates across all thresholds.
The dashed diagonal represents the random baseline (AUROC = 0.5).

---

## Qualitative Results

The final visualization shows side-by-side comparisons of:

- The original test image
- The ground truth defect mask (provided by MVTec)
- The predicted anomaly heatmap (continuous scores, color-coded)
- The predicted binary mask (after thresholding and morphological refinement)
This qualitative analysis complements the quantitative metrics by revealing failure
modes that aggregate numbers may obscure — for instance, cases where the model
detects the correct image region but overestimates the defect area, or cases where
a subtle defect produces a score only marginally above the threshold.

---

## Failure Analysis

The pipeline is expected to struggle in the following scenarios:

**Small defects** — `broken_small` fractures may occupy very few patches. If the
anomaly score is spread across neighboring normal patches after Gaussian smoothing,
the peak score may not be high enough to separate clearly from normal images.

**Contamination near the bottle boundary** — boundary regions are underrepresented
in the memory bank because the bottle occupies different spatial positions across
training images. Patches near edges may already have elevated distances even for
normal images, reducing the signal-to-noise ratio for contamination defects.

**Threshold sensitivity** — the 95th percentile threshold is applied per-image,
meaning every image produces a non-empty prediction regardless of its actual
anomaly score. For normal test images, this produces false positive masks even
when the image-level score is low. A global threshold calibrated on a held-out
validation set would mitigate this issue.

In [ ]:
# ── Compute image-level AUROC ─────────────────────────────────────────────────
# roc_auc_score() is the same scikit-learn function you already know!
# First argument:  test_labels   -> true labels (0=normal, 1=defective)
# Second argument: image_scores  -> predicted scores (continuous values)
# Returns the area under the ROC curve (between 0.5 and 1.0)
auroc_image = roc_auc_score(test_labels, image_scores)
print(f"Image-level AUROC: {auroc_image:.4f}")

In [ ]:
def pixel_metrics(gt_masks, pred_masks, labels):
    """Pixel-level IoU and Dice, evaluated only on defective images."""

    # Compute metrics only on defective images (label == 1)
    # labels == 1 creates a boolean mask used to select the corresponding rows
    mask = labels == 1
    # .astype(bool) converts 0/1 to False/True so we can use logical operators & (AND) and | (OR)
    gt   = gt_masks[mask].astype(bool)    # ground-truth masks for defective images
    pred = pred_masks[mask].astype(bool)  # predicted masks for defective images

    # & = pixel-wise logical AND: True only where BOTH masks are 1 (intersection)
    # | = pixel-wise logical OR:  True where AT LEAST ONE mask is 1 (union)
    # .sum() counts True pixels (equivalent to counting "white" pixels in the mask)
    intersect  = (gt & pred).sum()
    union      = (gt | pred).sum()

    # IoU (Intersection over Union) = |intersection| / |union|
    # float() converts the NumPy result to a standard Python number
    iou  = float(intersect / (union + 1e-8))

    # Dice = 2 * |intersection| / (|predicted| + |ground truth|)
    dice = float(2 * intersect / (gt.sum() + pred.sum() + 1e-8))

    return iou, dice


iou, dice = pixel_metrics(test_masks_gt, pred_masks, test_labels)

# ── Print the final metrics summary ───────────────────────────────────────────
print("─────────────────────────────────────")
print(f"  Image-level AUROC : {auroc_image:.4f}")
print(f"  Pixel-level IoU   : {iou:.4f}")
print(f"  Pixel-level Dice  : {dice:.4f}")
print("─────────────────────────────────────")

In [ ]:
# ── Compute the ROC curve ─────────────────────────────────────────────────────
# roc_curve() returns three arrays:
# fpr: False Positive Rate (false alarms) at each possible threshold
# tpr: True Positive Rate = Recall (defects correctly detected) at each threshold
# _  : the thresholds used (not needed here)
# By sweeping the threshold from 0 to +inf we get all points on the curve
fpr, tpr, _ = roc_curve(test_labels, image_scores)

plt.figure(figsize=(6, 6))

# Plot the ROC curve: X axis = FPR (false alarms), Y axis = TPR (sensitivity)
# lw=2: line width
# label: legend text; we include the already-computed AUROC
plt.plot(fpr, tpr, color="steelblue", lw=2,
         label=f"ROC curve  (AUROC = {auroc_image:.3f})")

# Dashed diagonal: random baseline (AUROC = 0.5, random classifier)
plt.plot([0, 1], [0, 1], "k--", lw=1, label="Random baseline")

# fill_between() shades the area under the ROC curve
# alpha=0.1: nearly transparent, just to highlight the area
plt.fill_between(fpr, tpr, alpha=0.1, color="steelblue")

plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate (Recall)")
plt.title("ROC Curve — Image-level Anomaly Detection")
plt.legend(loc="lower right")  # legend position
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# ── Comparison grid: normal vs defective images ───────────────────────────────
# Create a 2x4 grid: row 0 = normal images, row 1 = defective images
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
fig.suptitle("Detection Results", fontsize=14, fontweight="bold")

# Select 4 normal and 4 defective images from the test set
# np.where() returns the indices where the condition is True
# [0] picks the first array from the tuple returned by where; [:4] takes the first 4 elements
normal_idx = np.where(test_labels == 0)[0][:4]
defect_idx = np.where(test_labels == 1)[0][:4]

# zip() pairs normal and defective indices element-by-element
# enumerate() adds the column number: col=0,1,2,3
for col, (ni, di) in enumerate(zip(normal_idx, defect_idx)):

    # For each column display 2 images: row 0 = normal, row 1 = defective
    for row, idx in enumerate([ni, di]):
        img_t, label, _ = test_dataset[idx]  # _ discards the mask

        # Set caption text and colour based on the label
        caption = "Normal"    if label == 0 else "Defective"
        color   = "green"     if label == 0 else "red"

        axes[row, col].imshow(denormalize(img_t))
        # Title: category + anomaly score (:.2f = 2 decimal places)
        axes[row, col].set_title(f"{caption}\nscore={image_scores[idx]:.2f}", color=color)
        axes[row, col].axis("off")

plt.tight_layout()
plt.show()